# 30 — Silver SCD Type 2: counterparty history
Rebuilds `silver.party_history` from **all** valid Bronze versions of each party (deterministic and
idempotent). A new version starts when `risk_rating` or `country` changes.

* `eff_start_date` — first version: the party's creation date; later versions: date of the change
* `eff_end_date` — **exclusive**: the next version's start; `9999-12-31` for the current version

A snapshot for date *D* uses the version where `eff_start_date <= D < eff_end_date`.

In [ ]:
%run ./_common

In [ ]:
bronze = spark.table(fq("bronze", "PARTY"))
valid, _ = split_valid(bronze, get_rules("PARTY"))

# one row per (party, updated_at)
w_ver = Window.partitionBy("party_id", "updated_at").orderBy(F.col("ingested_at").desc())
versions = valid.withColumn("_rn", F.row_number().over(w_ver)).where("_rn = 1").drop("_rn")

w_first = Window.partitionBy("party_id").orderBy("updated_at")
versions = (versions
            .withColumn("_is_first", F.row_number().over(w_first) == 1)
            .withColumn("eff_start_date", F.when(F.col("_is_first"), F.to_date("created_at")).otherwise(F.to_date("updated_at"))))

# several changes on the same day collapse to the last one
w_day = Window.partitionBy("party_id", "eff_start_date").orderBy(F.col("updated_at").desc())
versions = versions.withColumn("_rn", F.row_number().over(w_day)).where("_rn = 1").drop("_rn")

# keep only rows where a tracked attribute actually changed
w_seq = Window.partitionBy("party_id").orderBy("eff_start_date")
versions = (versions
            .withColumn("_prev_rating", F.lag("risk_rating").over(w_seq))
            .withColumn("_prev_country", F.lag("country").over(w_seq))
            .where("_prev_rating IS NULL OR _prev_rating <> risk_rating OR _prev_country <> country"))

w_end = Window.partitionBy("party_id").orderBy("eff_start_date")
history = (versions
           .withColumn("eff_end_date", F.coalesce(F.lead("eff_start_date").over(w_end), F.lit("9999-12-31").cast("date")))
           .withColumn("is_current", F.col("eff_end_date") == F.lit("9999-12-31").cast("date"))
           .withColumn("version_no", F.row_number().over(w_end))
           .select("party_id", "party_name", "country", "party_type", "risk_rating",
                   "eff_start_date", "eff_end_date", "is_current", "version_no"))

history.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("silver", "party_history"))
print("party_history rows:", spark.table(fq("silver", "party_history")).count())

In [ ]:
display(spark.table(fq("silver", "party_history")).where("version_no > 1").orderBy("party_id", "version_no").limit(20))